# Predicción de Precios de Acciones con Redes Neuronales Profundas

In [6]:
# Importar librerías
import time
import numpy as np
import pandas as pd
import yfinance as yf
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import plotly.graph_objects as go
from plotly.subplots import make_subplots



In [7]:
# Descargar datos
df = yf.download("EURUSD=X", start="2010-01-01", end="2024-01-01", interval="1d")

# Si viene con MultiIndex, aplastar columnas
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Ahora las columnas deberían ser planas: Open, High, Low, Close, Adj Close, Volume
print(df.columns)


C:\Users\john_\AppData\Local\Temp\ipykernel_30732\1140568038.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("EURUSD=X", start="2010-01-01", end="2024-01-01", interval="1d")
[*********************100%***********************]  1 of 1 completed

Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')


In [8]:
print(df.columns)

Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')


In [9]:
df

Price,Close,High,Low,Open,Volume
Date,,,,,
2010-01-01,1.438994,1.440196,1.432706,1.432706,0
2010-01-04,1.442398,1.445191,1.426208,1.431004,0
2010-01-05,1.436596,1.448310,1.435194,1.442710,0
2010-01-06,1.440403,1.443460,1.429123,1.436596,0
2010-01-07,1.431803,1.444481,1.430206,1.440300,0
...,...,...,...,...,...
2023-12-25,1.102657,1.104240,1.099989,1.102657,0
2023-12-26,1.102026,1.103997,1.100958,1.102026,0
2023-12-27,1.104301,1.112248,1.102925,1.104301,0


In [ ]:
# Graficar Datos
fig = make_subplots(rows=2, cols=1, row_heights=[0.80, 0.20])

fig.add_trace(go.Candlestick(x=df.index, open=df["Open"], low=df["Low"], close=df["Close"], high=df["High"], name="EURUSD"),
             row=1, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df["Volume"], name="Volumen"), row=2, col=1)

fig.update_layout(height=600, width=1000, title="Gráfico de Velas y Volumen", xaxis_rangeslider_visible=False)
fig.show()

## Agregar Indicadores Técnicos

### Promedios Móviles

In [11]:
# Agregar diferentes Promedios Móviles
ventanas = [9, 10, 14, 20, 21, 50]
for i in ventanas:
    df[f"MA_{i}"] = df["Close"].rolling(i, min_periods=i).mean()
    
# Datos con las nuevas variables
df

Price,Close,High,Low,Open,Volume,MA_9,MA_10,MA_14,MA_20,MA_21,MA_50
Date,,,,,,,,,,,
2010-01-01,1.438994,1.440196,1.432706,1.432706,0,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-04,1.442398,1.445191,1.426208,1.431004,0,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-05,1.436596,1.448310,1.435194,1.442710,0,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-06,1.440403,1.443460,1.429123,1.436596,0,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-07,1.431803,1.444481,1.430206,1.440300,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2023-12-25,1.102657,1.104240,1.099989,1.102657,0,1.093967,1.092236,1.088156,1.089489,1.089706,1.078230
2023-12-26,1.102026,1.103997,1.100958,1.102026,0,1.096429,1.094773,1.089775,1.089799,1.090086,1.079162
2023-12-27,1.104301,1.112248,1.102925,1.104301,0,1.098172,1.097216,1.091748,1.089984,1.090489,1.080102


In [12]:
# Graficar
fig = go.Figure()
for i in ventanas:
    fig.add_trace(go.Scatter(x=df.index, y=df[f"MA_{i}"], name=f"Promedio Móvil {i}"))
# Agregar el precio de Cierre
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close", opacity=0.5))
fig.update_layout(title="Diferentes Promedios Móviles")
fig.show()

### Indicador RSI

In [13]:
# Implementar indicador
def Relative_Strength_Index(df: pd.DataFrame, longitud: int = 14) -> pd.Series:
    
    """
    El índice de Fuerza Relativa (RSI) es un indicador de impulso utilizado en el análisis técnico que mide
    la magnitud de los cambios recientes en los precios para evaluar las condiciones de sobrecompra o sobreventa
    en el precio de una acción u otro activo financiero.
    """
    
    # Calcular
    Delta = df["Close"].diff(periods=1)
    Gain = Delta.where(Delta  >= 0, 0)
    Loss = np.abs(Delta.where(Delta < 0, 0))
    avg_gain = Gain.ewm(alpha=1/longitud, min_periods=longitud).mean()
    avg_loss = Loss.ewm(alpha=1/longitud, min_periods=longitud).mean()
    RS = avg_gain / avg_loss
    RSI = pd.Series(np.where(RS == 0, 100, 100 - (100 / (1+RS))), name="RSI", index=df.index)
    
    return RSI

In [14]:
# Agregar el indicador a nuestros datos
ventanas = [9, 10, 14, 20, 21, 50]
for i in ventanas:
    df[f"RSI_{i}"] = Relative_Strength_Index(df, longitud=i)
    
fig = make_subplots(rows=3, cols=2)
fig.add_trace(go.Scatter(x=df.index, y=df["RSI_9"], name="RSI 9"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["RSI_10"], name="RSI 10"), row=1, col=2)
fig.add_trace(go.Scatter(x=df.index, y=df["RSI_14"], name="RSI 14"), row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["RSI_20"], name="RSI 20"), row=2, col=2)
fig.add_trace(go.Scatter(x=df.index, y=df["RSI_21"], name="RSI 21"), row=3, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["RSI_50"], name="RSI 50"), row=3, col=2)

fig.update_layout(title="Análsis de RSI para diferentes ventanas de tiempo")
fig.show()

### Indicador MACD

In [15]:
# Calcular indicador
df["EMA_12"] = pd.Series(df["Close"].ewm(span=12, min_periods=12).mean())
df["EMA_26"] = pd.Series(df["Close"].ewm(span=26, min_periods=26).mean())
df["MACD"] = df["EMA_12"] - df["EMA_26"]
df["MACD_signal"] = df["MACD"].ewm(span=9, min_periods=9).mean()

# Realizar gráfico
fig = make_subplots(rows=2, cols=1)
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Close"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["EMA_12"], name="EMA_12"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["EMA_26"], name="EMA_26"), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["MACD"], name="MACD"), row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df["MACD_signal"], name="MACD_Signal"), row=2, col=1)
fig.show()

## Limpiar Datos y Ajustarlos

In [16]:
# Desplazar 1 posición para predecir con los datos de un día anterior (excepto el Close)
Close = df["Close"]
df = df.shift(periods=1)
df["Prev_Close"] = df["Close"]
df["Close"] = Close
# Eliminar valores faltantes
df.dropna(inplace=True)
df

Price,Close,High,Low,Open,Volume,MA_9,MA_10,MA_14,MA_20,MA_21,...,RSI_10,RSI_14,RSI_20,RSI_21,RSI_50,EMA_12,EMA_26,MACD,MACD_signal,Prev_Close
Date,,,,,,,,,,,,,,,,,,,,,
2010-03-12,1.376993,1.368794,1.362305,1.364927,0.0,1.362776,1.362809,1.360662,1.360906,1.361252,...,52.767505,48.688124,45.517604,45.170261,41.449952,1.363250,1.368409,-0.005159,-0.007793,1.368532
2010-03-15,1.367746,1.379405,1.367353,1.368495,0.0,1.365085,1.364198,1.361836,1.361581,1.361672,...,59.409535,53.703620,49.330828,48.854202,43.754799,1.365365,1.369058,-0.003693,-0.006970,1.376993
2010-03-16,1.376879,1.377391,1.364592,1.377391,0.0,1.365717,1.365351,1.362910,1.361953,1.361875,...,50.744110,48.162741,45.654745,45.357028,41.914697,1.365731,1.368959,-0.003228,-0.006220,1.367746
2010-03-17,1.373834,1.378284,1.366102,1.367372,0.0,1.366463,1.366833,1.364522,1.361962,1.362664,...,57.540685,53.289046,49.562461,49.133307,44.276556,1.367446,1.369556,-0.002109,-0.005396,1.376879
2010-03-18,1.361396,1.381654,1.373098,1.376898,0.0,1.368178,1.367200,1.365967,1.362619,1.362527,...,54.742144,51.461574,48.342387,47.972573,43.672345,1.368429,1.369878,-0.001448,-0.004605,1.373834
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-25,1.102657,1.104057,1.099421,1.100619,0.0,1.091078,1.089628,1.086807,1.089059,1.089134,...,63.321618,61.773704,60.291880,60.071350,55.511969,1.091399,1.086860,0.004539,0.003611,1.100619
2023-12-26,1.102026,1.104240,1.099989,1.102657,0.0,1.093967,1.092236,1.088156,1.089489,1.089706,...,64.959228,63.037087,61.249740,60.993949,55.979497,1.093131,1.088030,0.005101,0.003909,1.102657
2023-12-27,1.104301,1.103997,1.100958,1.102026,0.0,1.096429,1.094773,1.089775,1.089799,1.090086,...,63.975540,62.349267,60.771458,60.538713,55.794053,1.094499,1.089067,0.005432,0.004214,1.102026


## Dividir Datos en Entrenamiento y Prueba

In [17]:
tamaño_prueba = 0.10

# Índices de división para el conjunto de datos
indice_division_prueba = int(df.shape[0] * (1 - tamaño_prueba))

# Conjuntos de datos de entrenamiento, validación y prueba
conjunto_entrenamiento = df.iloc[:indice_division_prueba].copy()
conjunto_prueba = df.iloc[indice_division_prueba+1:].copy()

# Gráficar
fig = go.Figure()
fig.add_trace(go.Scatter(x=conjunto_entrenamiento.index, y=conjunto_entrenamiento["Close"], name="Entrenamiento"))
fig.add_trace(go.Scatter(x=conjunto_prueba.index,  y=conjunto_prueba["Close"],  name="Prueba"))
fig.update_layout(title="División de Datos")
fig.show()

## Dividir en Características y Etiquetas

In [18]:
X_entrenamiento = conjunto_entrenamiento.drop(columns=["Close"])
y_entrenamiento = conjunto_entrenamiento["Close"]

X_prueba = conjunto_prueba.drop(columns=["Close"])
y_prueba =conjunto_prueba["Close"]

In [19]:
X_entrenamiento

Price,High,Low,Open,Volume,MA_9,MA_10,MA_14,MA_20,MA_21,MA_50,...,RSI_10,RSI_14,RSI_20,RSI_21,RSI_50,EMA_12,EMA_26,MACD,MACD_signal,Prev_Close
Date,,,,,,,,,,,,,,,,,,,,,
2010-03-12,1.368794,1.362305,1.364927,0.0,1.362776,1.362809,1.360662,1.360906,1.361252,1.392265,...,52.767505,48.688124,45.517604,45.170261,41.449952,1.363250,1.368409,-0.005159,-0.007793,1.368532
2010-03-15,1.379405,1.367353,1.368495,0.0,1.365085,1.364198,1.361836,1.361581,1.361672,1.391025,...,59.409535,53.703620,49.330828,48.854202,43.754799,1.365365,1.369058,-0.003693,-0.006970,1.376993
2010-03-16,1.377391,1.364592,1.377391,0.0,1.365717,1.365351,1.362910,1.361953,1.361875,1.389532,...,50.744110,48.162741,45.654745,45.357028,41.914697,1.365731,1.368959,-0.003228,-0.006220,1.367746
2010-03-17,1.378284,1.366102,1.367372,0.0,1.366463,1.366833,1.364522,1.361962,1.362664,1.388338,...,57.540685,53.289046,49.562461,49.133307,44.276556,1.367446,1.369556,-0.002109,-0.005396,1.376879
2010-03-18,1.381654,1.373098,1.376898,0.0,1.368178,1.367200,1.365967,1.362619,1.362527,1.387006,...,54.742144,51.461574,48.342387,47.972573,43.672345,1.368429,1.369878,-0.001448,-0.004605,1.373834
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-08-08,1.024905,1.014363,1.024779,0.0,1.019854,1.019871,1.019768,1.015844,1.015899,1.038640,...,53.686051,50.203224,47.312498,46.992054,43.648462,1.019888,1.024086,-0.004197,-0.006225,1.024779
2022-08-09,1.022181,1.016002,1.017087,0.0,1.019253,1.019577,1.019951,1.015868,1.015904,1.037519,...,45.996567,44.920835,43.761668,43.626523,42.275151,1.019457,1.023567,-0.004110,-0.005802,1.017087
2022-08-10,1.024695,1.018921,1.019763,0.0,1.020009,1.019304,1.019710,1.016619,1.016054,1.036367,...,48.829501,47.009668,45.265890,45.063671,42.912734,1.019504,1.023286,-0.003781,-0.005398,1.019763


## Construir Red Neuronal

In [20]:
# Definir semilla
torch.manual_seed(1)
np.random.seed(1)

# Escalar los datos
scaler = StandardScaler()
X_entrenamiento = scaler.fit_transform(X_entrenamiento)
X_prueba = scaler.transform(X_prueba)

# Covertir a tensores
X_entrenamiento_tensor = torch.tensor(X_entrenamiento, dtype=torch.float32)
y_entrenamiento_tensor = torch.tensor(y_entrenamiento.values, dtype=torch.float32).reshape(-1, 1)

X_prueba_tensor = torch.tensor(X_prueba, dtype=torch.float32)
y_prueba_tensor = torch.tensor(y_prueba.values, dtype=torch.float32).reshape(-1, 1)

In [21]:
# Definir arquitectura de red neuronal profunda
modelo = nn.Sequential(
    nn.Linear(in_features=X_entrenamiento_tensor.shape[1], out_features=X_entrenamiento_tensor.shape[1] * 4),
    nn.ReLU(),
    nn.Linear(in_features=X_entrenamiento_tensor.shape[1] * 4, out_features=X_entrenamiento_tensor.shape[1] * 4),
    nn.ReLU(),
    nn.Linear(in_features=X_entrenamiento_tensor.shape[1] * 4, out_features=X_entrenamiento_tensor.shape[1] * 4),
    nn.ReLU(),
    nn.Linear(in_features=X_entrenamiento_tensor.shape[1] * 4, out_features=1)
)
modelo

Sequential(
  (0): Linear(in_features=21, out_features=84, bias=True)
  (1): ReLU()
  (2): Linear(in_features=84, out_features=84, bias=True)
  (3): ReLU()
  (4): Linear(in_features=84, out_features=84, bias=True)
  (5): ReLU()
  (6): Linear(in_features=84, out_features=1, bias=True)
)

In [22]:
# Probar que funciona
modelo(X_entrenamiento_tensor)

tensor([[0.0456],
        [0.0533],
        [0.0493],
        ...,
        [0.0561],
        [0.0571],
        [0.0730]], grad_fn=<AddmmBackward0>)

In [23]:
# Definir función de pérdida y optimizador
lossfunc = nn.MSELoss()
optimizer = optim.Adam(modelo.parameters(), lr=0.001)

# Guardar valores pérdida
loss_values = []

# Entrenamiento del modelo
num_epochs = 3_000
for epoch in range(num_epochs):
    # Predecir y obtener el error
    outputs = modelo(X_entrenamiento_tensor)
    loss = lossfunc(outputs, y_entrenamiento_tensor)
    
    # Paso backward y optimización
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Calcular y almacenar la pérdida
    loss_values.append(loss.item())
    
    # Imprimir pérdida
    if (epoch + 1) % 100 == 0:
        print(f"Epoch: [{epoch + 1} / {num_epochs}], Loss: {loss.item():.4f}")

Epoch: [100 / 3000], Loss: 0.0027
Epoch: [200 / 3000], Loss: 0.0003
Epoch: [300 / 3000], Loss: 0.0002
Epoch: [400 / 3000], Loss: 0.0001
Epoch: [500 / 3000], Loss: 0.0001
Epoch: [600 / 3000], Loss: 0.0001
Epoch: [700 / 3000], Loss: 0.0001
Epoch: [800 / 3000], Loss: 0.0001
Epoch: [900 / 3000], Loss: 0.0001
Epoch: [1000 / 3000], Loss: 0.0000
Epoch: [1100 / 3000], Loss: 0.0000
Epoch: [1200 / 3000], Loss: 0.0000
Epoch: [1300 / 3000], Loss: 0.0000
Epoch: [1400 / 3000], Loss: 0.0000
Epoch: [1500 / 3000], Loss: 0.0000
Epoch: [1600 / 3000], Loss: 0.0000
Epoch: [1700 / 3000], Loss: 0.0000
Epoch: [1800 / 3000], Loss: 0.0000
Epoch: [1900 / 3000], Loss: 0.0000
Epoch: [2000 / 3000], Loss: 0.0000
Epoch: [2100 / 3000], Loss: 0.0000
Epoch: [2200 / 3000], Loss: 0.0000
Epoch: [2300 / 3000], Loss: 0.0000
Epoch: [2400 / 3000], Loss: 0.0000
Epoch: [2500 / 3000], Loss: 0.0000
Epoch: [2600 / 3000], Loss: 0.0000
Epoch: [2700 / 3000], Loss: 0.0000
Epoch: [2800 / 3000], Loss: 0.0000
Epoch: [2900 / 3000], Loss: 0

In [24]:
# Realizar las predicciones
y_entrenamiento_pred = modelo(X_entrenamiento_tensor).detach().numpy().flatten()
y_prueba_pred = modelo(X_prueba_tensor).detach().numpy().flatten()

In [25]:
# Gráficar
fig = go.Figure()
fig.add_trace(go.Scatter(x=conjunto_entrenamiento.index, y=conjunto_entrenamiento["Close"], name="Entrenamiento",
                         line=dict(color="red")))
fig.add_trace(go.Scatter(x=conjunto_entrenamiento.index, y=y_entrenamiento_pred, name=" Pedicción de Entrenamiento",
                         line=dict(color="green")))
fig.add_trace(go.Scatter(x=conjunto_prueba.index, y=conjunto_prueba["Close"], name="Prueba",
                         line=dict(color="blue")))
fig.add_trace(go.Scatter(x=conjunto_prueba.index, y=y_prueba_pred, name="Predicción de Prueba",
                         line=dict(color="orange")))
fig.update_layout(title="Datos Originales vs Predicción")
fig.show()

In [26]:
# Calcular el error absoluto en el conjunto de entrenamiento y de prueba
mae_entrenamiento = mean_absolute_error(y_entrenamiento, y_entrenamiento_pred)
mae_prueba = mean_absolute_error(y_prueba, y_prueba_pred)

print(f"MAE en conjunto de entrenamiento = {mae_entrenamiento}")
print(f"MAE en conjunto de prueba = {mae_prueba}")

MAE en conjunto de entrenamiento = 0.0045292043346999164
MAE en conjunto de prueba = 0.012737457134597481


# Recordatorio:

    - Las Redes Neuronales Profundas poseen una capacidad extraordinaria para descifrar patrones intrincados en datos financieros, lo que facilita el pronóstico más preciso del precio del activo.